# Alpha 세부 탐색 + 정량 평가 노트북

이 노트북은 다음을 한 번에 수행합니다.

1. 1차 alpha 비교(`0.2`, `0.3`, `0.4`) 실행
2. 각 alpha run의 학습 요약(`val_loss`, `val_SI-SNR`) 수집
3. 각 alpha의 대표 체크포인트를 다시 불러와 `PESQ / STOI / SI-SNR` 재평가
4. 결과를 CSV / JSON / PNG로 저장하고 alpha-성능 그래프를 그리기

기본 가정:
- `A-soft`, `seed=0`, `military` 환경 기준
- 데이터는 `processed_data.tar.gz` 형태로 Drive에 저장되어 있음
- alpha 탐색은 `scripts/alpha_search.py`를 사용하고, 정량 평가는 이 노트북에서 추가로 수행함


---
## Step 0. 환경 설정

레포를 클론/업데이트하고 필요한 패키지를 설치합니다.


In [ ]:
import os, sys, subprocess, shlex

GITHUB_USER = 'sungmin-park-dev'
REPO_NAME   = 'tactical-speech-enhancement'
REPO_DIR    = f'/content/{REPO_NAME}'
DRIVE_ROOT  = '/content/drive/MyDrive/Colab Notebooks/ARMY Projects/2026-tactical-speech-enhancement'
COLAB_CFG   = '/content/data_config_colab.yaml'

if not os.path.isdir(REPO_DIR):
    token = os.getenv('GITHUB_TOKEN', '')
    if token:
        clone_url = f'https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    else:
        clone_url = f'https://github.com/{GITHUB_USER}/{REPO_NAME}.git'
    print(f'클론 중: github.com/{GITHUB_USER}/{REPO_NAME}')
    result = subprocess.run(
        ['git', 'clone', clone_url, REPO_DIR],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f'git clone 실패 (exit {result.returncode})')
        print(result.stderr.strip())
        if not token:
            print('Private 레포면 Runtime -> Manage secrets -> GITHUB_TOKEN 을 설정하세요.')
        raise RuntimeError('git clone 실패')
    print('레포 클론 완료')
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--quiet'], check=True)
    print('레포 최신화 완료')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'soundfile', 'librosa', 'pesq', 'pystoi', 'pyyaml', 'tqdm',
        'matplotlib', 'pandas', 'seaborn'
    ],
    check=True,
)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
!df -h /content | tail -1


---
## Step 1. 데이터 복원

Drive의 `processed_data.tar.gz`를 `/content/data/processed`로 복원합니다.
기존 alpha 탐색 결과 archive가 있으면 선택적으로 같이 복원할 수 있습니다.


In [ ]:
import glob, yaml
from google.colab import drive

drive.mount('/content/drive')

DATA_DIR = '/content/data/processed'
RESULTS_DIR = '/content/results'
DRIVE_DATA_ARCHIVE = f'{DRIVE_ROOT}/data/processed_data.tar.gz'
DRIVE_RESULTS_ARCHIVE = f'{DRIVE_ROOT}/results/alpha_search_fine_latest.tar.gz'
RESTORE_RESULTS = False  # 기존 alpha 탐색 결과 archive가 있으면 True로 바꾸세요.

if os.path.exists(DATA_DIR):
    print('processed 데이터 이미 로컬에 있음 — 스킵')
elif os.path.exists(DRIVE_DATA_ARCHIVE):
    print('Drive에서 processed 데이터 복원 중...')
    !cp "{DRIVE_DATA_ARCHIVE}" /tmp/processed_data.tar.gz
    !mkdir -p /content/data && tar -xzf /tmp/processed_data.tar.gz -C /content/data/
    !rm /tmp/processed_data.tar.gz
    print('processed 데이터 복원 완료')
else:
    print('processed_data.tar.gz 가 없습니다. download_to_drive.ipynb를 먼저 실행하세요.')

if RESTORE_RESULTS and os.path.exists(DRIVE_RESULTS_ARCHIVE):
    print('Drive에서 alpha 결과 복원 중...')
    !mkdir -p /content
    !cp "{DRIVE_RESULTS_ARCHIVE}" /tmp/alpha_results.tar.gz
    !tar -xzf /tmp/alpha_results.tar.gz -C /content/
    !rm /tmp/alpha_results.tar.gz
    print('alpha 결과 복원 완료')
else:
    print('결과 복원 스킵 — /content/results 를 그대로 사용하거나 새로 탐색합니다.')

with open(f'{REPO_DIR}/configs/data_config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['output_root'] = DATA_DIR
with open(COLAB_CFG, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print('COLAB config:', COLAB_CFG)
!du -sh /content/data/processed/* 2>/dev/null || echo '데이터 없음'


---
## Step 2. Alpha 세부 탐색 실행

중요: 현재 `scripts/alpha_search.py`는 학습 중 기록되는 `best_val_sisnr` / `best_val_loss`를 모읍니다.
즉, **PESQ / STOI 비교는 아직 포함하지 않으며**, 그 부분은 아래 Step 4에서 이 노트북이 추가로 계산합니다.

`RUN_TRAINING=False`로 바꾸면 기존 결과만 재분석할 수 있습니다.


In [ ]:
import subprocess, torch

ENV = 'military'
ABLATION = 'A-soft'
SEED = 0
EPOCHS = 50
BATCH = 16 if 'A100' in (torch.cuda.get_device_name(0) if torch.cuda.is_available() else '') else 8

SEARCH_MODE = 'explicit'   # 'explicit' or 'grid'
ALPHAS = [0.20, 0.30, 0.40]
CENTER = 0.30
RADIUS = 0.10
STEP = 0.10

SAVE_ROOT = '/content/results/dpcrn/alpha_search_fine'
SKIP_EXISTING = True
RUN_TRAINING = True

cmd = [
    sys.executable,
    '-u',
    f'{REPO_DIR}/scripts/alpha_search.py',
    '--config', f'{REPO_DIR}/configs/train_config.yaml',
    '--data_config', COLAB_CFG,
    '--ablation', ABLATION,
    '--env', ENV,
    '--seed', str(SEED),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH),
    '--save_root', SAVE_ROOT,
    '--selection_metric', 'best_val_sisnr',
]

if SKIP_EXISTING:
    cmd.append('--skip_existing')

if SEARCH_MODE == 'explicit':
    cmd.extend(['--alphas', *[str(a) for a in ALPHAS]])
else:
    cmd.extend(['--center', str(CENTER), '--radius', str(RADIUS), '--step', str(STEP)])

print('실행 명령:')
print(shlex.join(cmd))

if RUN_TRAINING:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end='')
    finally:
        if proc.stdout is not None:
            proc.stdout.close()
    return_code = proc.wait()
    if return_code != 0:
        raise RuntimeError(f'alpha_search.py failed with exit code {return_code}')
else:
    print('RUN_TRAINING=False 이므로 학습은 실행하지 않고 기존 결과만 분석합니다.')


---
## Step 3. Alpha 탐색 결과 수집

새 버전의 `train.py`가 만든 `metrics_summary.json`을 우선 사용하고,
없으면 체크포인트(`best_sisnr.pt` / `best.pt`)에서 가능한 필드를 fallback으로 읽습니다.


In [ ]:
import json, re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

from data.dataset import SpeechEnhancementDataset
from models.dpcrn_dual import DPCRNDual
from evaluate import compute_metrics

ALPHA_ROOT = Path(SAVE_ROOT)

def parse_alpha_value(path):
    match = re.search(r'alpha([0-9]+(?:\.[0-9]+)?)', Path(path).name)
    if match:
        return float(match.group(1))
    return float('nan')

def choose_checkpoint(run_dir, preferred='best_sisnr.pt'):
    run_dir = Path(run_dir)
    for name in [preferred, 'best.pt', 'last.pt']:
        ckpt_path = run_dir / name
        if ckpt_path.exists():
            return ckpt_path
    return None

def load_training_row(run_dir):
    run_dir = Path(run_dir)
    summary_path = run_dir / 'metrics_summary.json'
    alpha_guess = parse_alpha_value(run_dir)

    if summary_path.exists():
        with open(summary_path, 'r', encoding='utf-8') as f:
            row = json.load(f)
        row['alpha'] = float(row.get('alpha', alpha_guess))
        row['run_dir'] = str(run_dir)
        row['checkpoint_path'] = str(choose_checkpoint(run_dir) or '')
        return row

    ckpt_path = choose_checkpoint(run_dir, preferred='best.pt')
    if ckpt_path is None:
        return None

    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    best_epoch_by_loss = ckpt.get('best_epoch_by_loss')
    best_epoch_by_sisnr = ckpt.get('best_epoch_by_sisnr')
    epoch = ckpt.get('epoch')

    return {
        'alpha': float(ckpt.get('alpha', alpha_guess)),
        'seed': ckpt.get('seed'),
        'env': ckpt.get('env'),
        'ablation_id': ckpt.get('ablation_id', ABLATION),
        'mask_type': ckpt.get('mask_type', 'soft'),
        'best_val_loss': ckpt.get('best_val_loss', ckpt.get('val_loss')),
        'best_epoch_by_loss': (best_epoch_by_loss + 1) if isinstance(best_epoch_by_loss, int) and best_epoch_by_loss >= 0 else None,
        'best_val_sisnr': ckpt.get('best_val_sisnr', ckpt.get('val_sisnr')),
        'best_epoch_by_sisnr': (best_epoch_by_sisnr + 1) if isinstance(best_epoch_by_sisnr, int) and best_epoch_by_sisnr >= 0 else None,
        'last_val_loss': ckpt.get('val_loss'),
        'last_val_sisnr': ckpt.get('val_sisnr'),
        'epochs_completed': (epoch + 1) if isinstance(epoch, int) else None,
        'save_dir': str(run_dir),
        'run_dir': str(run_dir),
        'checkpoint_path': str(ckpt_path),
    }

def scan_alpha_runs(alpha_root):
    alpha_root = Path(alpha_root)
    rows = []
    if not alpha_root.exists():
        return pd.DataFrame()

    for run_dir in sorted(alpha_root.iterdir()):
        if not run_dir.is_dir():
            continue
        if 'alpha' not in run_dir.name:
            continue
        row = load_training_row(run_dir)
        if row is not None:
            rows.append(row)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values('alpha').reset_index(drop=True)
    for col in ['alpha', 'best_val_loss', 'best_val_sisnr', 'last_val_loss', 'last_val_sisnr']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def build_model_from_checkpoint(ckpt, device):
    cfg = ckpt['config']
    ablation_id = ckpt.get('ablation_id', ABLATION)
    model = DPCRNDual(
        n_fft=cfg['audio']['n_fft'],
        hop_length=cfg['audio']['hop_length'],
        win_length=cfg['audio']['win_length'],
        ablation_id=ablation_id,
        n_dual_path_blocks=cfg['model']['n_dual_path_blocks'],
        lstm_hidden=cfg['model']['lstm_hidden'],
        use_bwe=cfg['model'].get('use_bwe', False),
    ).to(device)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model, ablation_id

def summarize_metric_dict(metric_lists):
    out = {}
    for name, values in metric_lists.items():
        arr = np.array(values, dtype=np.float64)
        out[f'{name}_mean'] = float(np.nanmean(arr))
        out[f'{name}_std'] = float(np.nanstd(arr))
    return out

def compute_noisy_baseline(dataset, sample_indices, sample_rate=16000):
    metric_lists = {'si_snr': [], 'pesq': [], 'stoi': []}
    for idx in tqdm(sample_indices, desc='Noisy baseline', leave=False):
        sample = dataset[idx]
        noisy_np = sample['ac'].numpy()
        clean_np = sample['clean'].numpy()
        min_len = min(len(noisy_np), len(clean_np))
        metric = compute_metrics(noisy_np[:min_len], clean_np[:min_len], sample_rate=sample_rate)
        for key in metric_lists:
            metric_lists[key].append(metric[key])
    return summarize_metric_dict(metric_lists)

def evaluate_run(ckpt_path, dataset, sample_indices, device, sample_rate=16000):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model, ablation_id = build_model_from_checkpoint(ckpt, device)
    metric_lists = {'si_snr': [], 'pesq': [], 'stoi': []}

    with torch.no_grad():
        for idx in tqdm(sample_indices, desc=f'eval {Path(ckpt_path).parent.name}', leave=False):
            sample = dataset[idx]
            bc = sample['bc'].unsqueeze(0).to(device)
            ac = sample['ac'].unsqueeze(0).to(device)
            clean_np = sample['clean'].numpy()
            mask = sample['mask'].unsqueeze(0).to(device) if ablation_id in ('A-soft', 'A-hard', 'A-param') else None
            ac_in = ac if ablation_id != 'C' else None

            enhanced_np = model(bc, ac_in, mask).squeeze(0).cpu().numpy()
            min_len = min(len(enhanced_np), len(clean_np))
            metric = compute_metrics(enhanced_np[:min_len], clean_np[:min_len], sample_rate=sample_rate)
            for key in metric_lists:
                metric_lists[key].append(metric[key])

    return summarize_metric_dict(metric_lists)


In [ ]:
train_df = scan_alpha_runs(ALPHA_ROOT)
if train_df.empty:
    raise RuntimeError(f'alpha 결과를 찾지 못했습니다: {ALPHA_ROOT}')

display_cols = [
    'alpha', 'best_val_loss', 'best_val_sisnr',
    'best_epoch_by_loss', 'best_epoch_by_sisnr', 'checkpoint_path'
]
display(train_df[display_cols])

best_train_row = train_df.loc[train_df['best_val_sisnr'].idxmax()]
print(f"학습 중 val_SI-SNR 기준 최적 alpha = {best_train_row['alpha']:.3f}")
print(f"학습 중 best val_SI-SNR = {best_train_row['best_val_sisnr']:.2f} dB")


---
## Step 4. PESQ / STOI / SI-SNR 재평가

여기서부터는 각 alpha의 체크포인트를 실제로 다시 불러와 동일한 데이터셋에서 정량 평가합니다.
기본은 `best_sisnr.pt`를 우선 사용하고, 없으면 `best.pt`로 fallback 합니다.

처음에는 `SPLIT='val'`, `N_SAMPLES=50` 정도로 빠르게 보고,
최종 비교 단계에서는 `SPLIT='test'` 또는 더 큰 `N_SAMPLES`로 바꾸는 것을 권장합니다.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

SPLIT = 'val'      # 'val' 또는 'test'
N_SAMPLES = 50     # 시간이 오래 걸리면 줄이세요
CKPT_NAME = 'best_sisnr.pt'

ANALYSIS_DIR = ALPHA_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

dataset = SpeechEnhancementDataset(
    data_dir=DATA_DIR,
    env=ENV,
    split=SPLIT,
    mask_type='soft',
)
sample_indices = list(range(min(N_SAMPLES, len(dataset))))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'{ENV}/{SPLIT} 데이터 {len(dataset)}개 중 {len(sample_indices)}개 평가')

baseline = compute_noisy_baseline(dataset, sample_indices)
print('Noisy baseline:', baseline)

eval_rows = []
for row in train_df.to_dict('records'):
    run_dir = Path(row['run_dir'])
    ckpt_path = choose_checkpoint(run_dir, preferred=CKPT_NAME)
    if ckpt_path is None:
        print(f'체크포인트 없음: {run_dir}')
        continue

    print(f"평가 중: alpha={row['alpha']:.3f} -> {ckpt_path.name}")
    metrics = evaluate_run(ckpt_path, dataset, sample_indices, device)
    eval_rows.append({
        'alpha': row['alpha'],
        'eval_ckpt': ckpt_path.name,
        **metrics,
    })

eval_df = pd.DataFrame(eval_rows).sort_values('alpha').reset_index(drop=True)
if eval_df.empty:
    raise RuntimeError('정량 평가 결과가 비어 있습니다.')

for metric in ['si_snr', 'pesq', 'stoi']:
    eval_df[f'delta_{metric}'] = eval_df[f'{metric}_mean'] - baseline[f'{metric}_mean']

display(eval_df)


---
## Step 5. 결과 저장 + 그래프 그리기

학습 요약, 정량 평가 결과, merged summary, noisy baseline, 그래프를 모두 `analysis/` 아래에 저장합니다.


In [ ]:
merged_df = train_df.merge(eval_df, on='alpha', how='left')

train_csv = ANALYSIS_DIR / 'alpha_training_summary.csv'
eval_csv = ANALYSIS_DIR / 'alpha_eval_summary.csv'
merged_csv = ANALYSIS_DIR / 'alpha_merged_summary.csv'
baseline_json = ANALYSIS_DIR / 'alpha_noisy_baseline.json'
summary_json = ANALYSIS_DIR / 'alpha_summary.json'
plot_path = ANALYSIS_DIR / 'alpha_metric_curves.png'

train_df.to_csv(train_csv, index=False)
eval_df.to_csv(eval_csv, index=False)
merged_df.to_csv(merged_csv, index=False)

with open(baseline_json, 'w', encoding='utf-8') as f:
    json.dump(baseline, f, indent=2)

best_alpha_train = float(train_df.loc[train_df['best_val_sisnr'].idxmax(), 'alpha'])
best_alpha_si = float(merged_df.loc[merged_df['si_snr_mean'].idxmax(), 'alpha'])
best_alpha_pesq = float(merged_df.loc[merged_df['pesq_mean'].idxmax(), 'alpha'])
best_alpha_stoi = float(merged_df.loc[merged_df['stoi_mean'].idxmax(), 'alpha'])

summary_payload = {
    'split': SPLIT,
    'n_samples': len(sample_indices),
    'checkpoint_preference': CKPT_NAME,
    'best_alpha_by_train_val_sisnr': best_alpha_train,
    'best_alpha_by_eval_si_snr': best_alpha_si,
    'best_alpha_by_eval_pesq': best_alpha_pesq,
    'best_alpha_by_eval_stoi': best_alpha_stoi,
    'noisy_baseline': baseline,
}
with open(summary_json, 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=2)

display_cols = [
    'alpha', 'best_val_sisnr',
    'si_snr_mean', 'delta_si_snr',
    'pesq_mean', 'delta_pesq',
    'stoi_mean', 'delta_stoi',
]
display(merged_df[display_cols])

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(merged_df['alpha'], merged_df['best_val_sisnr'], marker='o', linewidth=2)
axes[0, 0].axvline(best_alpha_train, linestyle=':', color='tab:green', alpha=0.8)
axes[0, 0].set_title('Train-time best val_SI-SNR')
axes[0, 0].set_xlabel('alpha')
axes[0, 0].set_ylabel('dB')

metric_specs = [
    (axes[0, 1], 'si_snr_mean', 'Validation SI-SNR', 'dB', 'si_snr_mean'),
    (axes[1, 0], 'pesq_mean', 'Validation PESQ', 'score', 'pesq_mean'),
    (axes[1, 1], 'stoi_mean', 'Validation STOI', 'score', 'stoi_mean'),
]

for ax, metric_col, title, ylabel, baseline_key in metric_specs:
    best_alpha_metric = float(merged_df.loc[merged_df[metric_col].idxmax(), 'alpha'])
    ax.plot(merged_df['alpha'], merged_df[metric_col], marker='o', linewidth=2, label='Enhanced')
    ax.axhline(baseline[baseline_key], linestyle='--', color='tab:red', label='Noisy baseline')
    ax.axvline(best_alpha_metric, linestyle=':', color='tab:green', alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('alpha')
    ax.set_ylabel(ylabel)
    ax.legend()

fig.tight_layout()
fig.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.show()

print('저장 완료')
print('  train summary :', train_csv)
print('  eval summary  :', eval_csv)
print('  merged summary:', merged_csv)
print('  baseline json :', baseline_json)
print('  summary json  :', summary_json)
print('  plot png      :', plot_path)
print()
print(f'학습 중 val_SI-SNR 기준 최적 alpha: {best_alpha_train:.3f}')
print(f'재평가 SI-SNR 기준 최적 alpha:   {best_alpha_si:.3f}')
print(f'재평가 PESQ 기준 최적 alpha:     {best_alpha_pesq:.3f}')
print(f'재평가 STOI 기준 최적 alpha:     {best_alpha_stoi:.3f}')
